# Analyze gait `.npz` and `.npy` files

This notebook shows how to inspect saved NumPy artifacts from a gait training run.

- `.npz` is a zip-like container with named arrays, such as `gait_meta_*.npz`.
- `.npy` is one array, such as `gait_best_*.npy` or `gait_fitness_history_*.npy`.

The default paths below point at the run you mentioned. Change `RUN_DIR` or individual file paths to inspect another run.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "helper":
    ROOT = ROOT.parent

RUN_DIR = ROOT / "results/cmaes/20260609_132427_778702_seed42"
META_PATH = RUN_DIR / "gait_meta_20260609_132427_778702_seed42.npz"

print(ROOT)
print(META_PATH)

/Users/yizhou/bachlor_project
/Users/yizhou/bachlor_project/results/cmaes/20260609_132427_778702_seed42/gait_meta_20260609_132427_778702_seed42.npz


## Generic NumPy artifact inspection

Use `np.load(...)` for both `.npy` and `.npz`. For `.npz`, inspect `.files` to see the named entries.

In [2]:
def scalar_or_array(value):
    if isinstance(value, np.ndarray) and value.shape == ():
        return value.item()
    return value


def summarize_array(arr: np.ndarray) -> str:
    if arr.shape == ():
        return repr(arr.item())
    if arr.dtype.kind in "biufc" and arr.size:
        return (
            f"shape={arr.shape}, dtype={arr.dtype}, "
            f"min={np.nanmin(arr):.4g}, max={np.nanmax(arr):.4g}, "
            f"mean={np.nanmean(arr):.4g}"
        )
    preview = arr[:5] if arr.ndim and len(arr) else arr
    return f"shape={arr.shape}, dtype={arr.dtype}, preview={preview!r}"


def inspect_numpy_file(path: Path):
    path = Path(path)
    data = np.load(path, allow_pickle=True)
    print(path)
    print("-" * len(str(path)))
    if isinstance(data, np.lib.npyio.NpzFile):
        print(f"NPZ with {len(data.files)} keys")
        for key in data.files:
            print(f"{key:32s} {summarize_array(data[key])}")
        return data

    print("NPY array")
    print(summarize_array(data))
    return data


meta = inspect_numpy_file(META_PATH)

/Users/yizhou/bachlor_project/results/cmaes/20260609_132427_778702_seed42/gait_meta_20260609_132427_778702_seed42.npz
---------------------------------------------------------------------------------------------------------------------
NPZ with 27 keys
num_joints                       8
dt                               0.05
seed                             42
fitness                          'speed'
reach_radius                     0.15
population                       50
requested_population             50
num_actors                       15
used_ray                         True
budget                           200
duration                         30
sigma                            0.075
fall_z_threshold                 0.05
fall_tilt_threshold_deg          75.0
use_tilt_fall                    False
optimizer                        'nevergrad.ParametrizedCMA'
target_positions                 shape=(5, 3), dtype=float32, min=-3, max=0.1, mean=-0.6333
num_params                       

## Read metadata as plain Python values

Metadata files often store scalar NumPy arrays. This converts scalar entries to normal Python values for easier printing.

In [ ]:
meta_dict = {key: scalar_or_array(meta[key]) for key in meta.files}

for key, value in meta_dict.items():
    if isinstance(value, np.ndarray):
        print(f"{key:32s} array shape={value.shape}, dtype={value.dtype}")
    else:
        print(f"{key:32s} {value!r}")

## Load the linked best weights

`best_weights` in the metadata points to the `.npy` file containing the best optimized CPG vector.

In [ ]:
best_weights_path = ROOT / str(meta_dict["best_weights"])
best_weights = inspect_numpy_file(best_weights_path)

num_joints = int(meta_dict["num_joints"])
optimized_groups = [str(group) for group in meta_dict["optimized_cpg_groups"]]

print(f"num_joints={num_joints}")
print(f"optimized_groups={optimized_groups}")
print(f"expected vector length={num_joints * len(optimized_groups)}")
print(f"actual vector length={best_weights.size}")

## Decode a CPG vector by parameter group

Older runs used 4 groups: `phase`, `w`, `amplitudes`, `ha`. New fixed NA-CPG runs use 5 groups: `phase`, `w`, `amplitudes`, `ha`, `b`.

In [ ]:
def split_cpg_vector(vector: np.ndarray, groups: list[str], n_joints: int) -> dict[str, np.ndarray]:
    vector = np.asarray(vector, dtype=np.float32)
    expected = len(groups) * n_joints
    if vector.size != expected:
        raise ValueError(f"Vector has {vector.size} values, expected {expected}")
    return {
        group: vector[i * n_joints : (i + 1) * n_joints]
        for i, group in enumerate(groups)
    }


groups = split_cpg_vector(best_weights, optimized_groups, num_joints)
for name, values in groups.items():
    print(f"{name:12s}", np.round(values, 4))

In [ ]:
fig, axes = plt.subplots(len(groups), 1, figsize=(10, 2.2 * len(groups)), sharex=True)
if len(groups) == 1:
    axes = [axes]

x = np.arange(num_joints)
for ax, (name, values) in zip(axes, groups.items()):
    ax.bar(x, values)
    ax.set_ylabel(name)
    ax.grid(axis="y", alpha=0.3)

axes[-1].set_xlabel("actuator / CPG output index")
fig.suptitle("Best CPG parameters by group")
fig.tight_layout()
plt.show()

## Plot fitness history

The run folder stores per-generation population-best fitness and global-best fitness as `.npy` arrays.

In [ ]:
fitness_path = RUN_DIR / f"gait_fitness_history_{meta_dict['run_id']}.npy"
global_path = RUN_DIR / f"gait_global_best_history_{meta_dict['run_id']}.npy"

fitness = np.load(fitness_path) if fitness_path.exists() else None
global_best = np.load(global_path) if global_path.exists() else None

plt.figure(figsize=(10, 4))
if fitness is not None:
    plt.plot(fitness, label="population best")
if global_best is not None:
    plt.plot(global_best, label="global best")
plt.xlabel("generation")
plt.ylabel("fitness score lower is better")
plt.title("Gait optimization history")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

if global_best is not None:
    print(f"final global best: {global_best[-1]:.6g}")

## Compare checkpoints

This loads `checkpoints/*.npy`, decodes each vector by group, and plots how each group changes across saved generations.

In [ ]:
checkpoint_dir = RUN_DIR / "checkpoints"
checkpoint_paths = sorted(checkpoint_dir.glob("*.npy"))
print(f"found {len(checkpoint_paths)} checkpoints")
for path in checkpoint_paths:
    print(path.name)

In [ ]:
checkpoint_groups = {}
for path in checkpoint_paths:
    vector = np.load(path)
    checkpoint_groups[path.stem] = split_cpg_vector(vector, optimized_groups, num_joints)

for group_name in optimized_groups:
    plt.figure(figsize=(10, 3))
    for ckpt_name, split in checkpoint_groups.items():
        label = ckpt_name.split("_gen")[-1]
        plt.plot(split[group_name], marker="o", label=f"gen {label}")
    plt.title(group_name)
    plt.xlabel("actuator / CPG output index")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()

## Stability metrics

If the metadata points to a stability metrics `.npz`, inspect it the same way.

In [ ]:
stability_path = ROOT / str(meta_dict.get("stability_metrics", ""))
if stability_path.exists():
    stability = inspect_numpy_file(stability_path)
else:
    print(f"No stability metrics file found at {stability_path}")

## Quick saturation check for old checkpoints

Large amplitudes and high `speed` scaling can pin commands at `±pi/2`. This cell only checks static parameter magnitudes. To check actual command saturation, run the controller forward over time and log actions.

In [ ]:
if "amplitudes" in groups:
    amps = groups["amplitudes"]
    print("amplitudes:", np.round(amps, 4))
    print("abs(amplitudes) > pi/2:", np.where(np.abs(amps) > np.pi / 2)[0].tolist())
    print("abs(amplitudes) > pi/4:", np.where(np.abs(amps) > np.pi / 4)[0].tolist())

if "ha" in groups:
    ha = groups["ha"]
    print("ha:", np.round(ha, 4))
    print("abs(ha) >= 1 risky indices:", np.where(np.abs(ha) >= 1.0)[0].tolist())